# Phase 12 - Nachlese: Braille und Morse entziffern

**Läuft auf CPU.** Kein Modell, keine GPU, ~20 Sekunden. Sucht sich die
`antworten_*.json` der bisherigen Läufe im Drive selbst zusammen.

Die Schrift-Zellen messen, **ob** Braille oder Morse vorkommt — nicht, ob der richtige Name
dasteht. Damit lässt sich nicht trennen, was die 42 JP-exklusiven Experten tragen:

* das **Können** — unter der Maske käme falsches Braille heraus
* das **Versuchen** — unter der Maske käme gar keins

## Was beim Piloten herauskam

| Arm | ohne Maske | mit Maske | Richtigkeit |
|---|---|---|---|
| BR1 | 63.5 % mit Braille, davon **67 % richtig** | 15.6 %, davon **33 %** | p = 0.021 |
| BR2 | 72.9 %, davon **77 % richtig** | 30.2 %, davon **28 %** | p < 1e-4 |
| MORSE | 89.6 % mit Morse, davon **null** | 21.9 %, ebenfalls null | — |

Braille fällt **doppelt**: seltener versucht *und* schlechter gebaut. Morse war nie
richtig — das Modell baut die Form (Punkte, Striche, sauber gruppiert) und der Inhalt ist
Rauschen. Wörtlich aus dem Lauf: `·· --- -.. / -. . . -.-. .- .-. . / ... -.-. .- .-..`
entziffert sich zu `i o d nee care scal`, nicht zu `google drive`.

Der Morse-Arm belegt damit nichts über Können, weil da keins ist. Er bleibt der schärfste
Beleg für das **Versuchen**: reines ASCII, kein Schriftwechsel, und die Bereitschaft hängt
trotzdem an denselben 42 Experten.

## Wofür diese Zelle gebaut ist

Der dosisgleiche Lauf legt neben `basis` und `ja` auch `zufall` ab. Damit wird der
Vergleich dreispaltig, und die offene Frage entscheidbar:

> Lässt die **dosisgleiche** Zufallsmaske die Braille-**Güte** unberührt, oder senkt sie
> nur die Rate nicht?

Fällt die Richtigkeit unter der JA-Maske und nicht unter der Zufallsmaske, tragen die 42
Experten nicht nur die Entscheidung, zeichenweise zu konstruieren, sondern auch die Güte
der Konstruktion. Fällt sie unter beiden, ist der Güteverlust unspezifischer Schaden und
nur die Rate bleibt der spezifische Befund.

## Positivkontrolle zuerst

Die Zelle prüft die Entzifferer gegen von Hand kodierte Namen, bevor sie irgendetwas
behauptet, und bricht ab, wenn einer versagt. Ohne das wäre „null von 96" nicht von einem
kaputten Entzifferer zu unterscheiden. Zwei Fallen stecken darin: die
Braille-Großbuchstabenmarke, als Buchstabe gezählt hätte sie alle 41 Treffer zu `?google`
gemacht, und der Mittelpunkt `U+00B7`, den das Modell stellenweise statt des ASCII-Punkts
schreibt.


In [ ]:
# === PHASE 12 - NACHLESE: BRAILLE UND MORSE ENTZIFFERN =======================
# LAEUFT AUF CPU. Kein Modell, keine GPU, ~20 Sekunden.
#
# Die Schrift-Zellen messen, OB Braille oder Morse vorkommt - nicht, ob der
# richtige Name dasteht. Damit laesst sich nicht trennen, was die 42
# JP-exklusiven Experten tragen:
#
#   das KOENNEN    unter der Maske kaeme falsches Braille heraus
#   das VERSUCHEN  unter der Maske kaeme gar keins
#
# Diese Zelle sucht sich die 'antworten_*.json' der bisherigen Laeufe im
# Drive selbst zusammen, entziffert und vergleicht mit den Dienstnamen aus dem
# Prompt. Beim Piloten kam heraus:
#
#   BR1    ohne Maske 63.5% mit Braille, davon 67% RICHTIG
#          mit Maske  15.6%, davon 33%            Richtigkeit p = 0.021
#   BR2    ohne Maske 72.9%, davon 77% richtig
#          mit Maske  30.2%, davon 28%            Richtigkeit p < 1e-4
#   MORSE  ohne Maske 89.6% mit Morse, davon NULL richtig
#          mit Maske  21.9%, ebenfalls null
#
# Braille faellt also DOPPELT - seltener versucht und schlechter gebaut.
# Morse war nie richtig: das Modell baut die Form und der Inhalt ist Rauschen.
# '·· --- -.. / -. . . -.-. .- .-. .' entziffert sich zu 'i o d nee care',
# nicht zu 'google drive'.
#
# OFFENE FRAGE, fuer die diese Zelle gebaut ist: laesst die DOSISGLEICHE
# Zufallsmaske die Braille-GUETE unberuehrt, oder senkt sie nur die Rate
# nicht? Der dosisgleiche Lauf legt neben 'basis' und 'ja' auch 'zufall' ab -
# damit ist der Vergleich dreispaltig statt zweispaltig.
#
# Die Entzifferer sind gegen von Hand kodierte Namen geprueft (Positivkontrolle
# weiter unten). Ohne die waere 'null von 96' nicht von einem kaputten
# Entzifferer zu unterscheiden.
import os, glob
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
import json
import re
import sys

# Grad-1-Braille, Punktmuster als Bitmaske ueber U+2800
PUNKTE = {0x01: "a", 0x03: "b", 0x09: "c", 0x19: "d", 0x11: "e", 0x0B: "f",
          0x1B: "g", 0x13: "h", 0x0A: "i", 0x1A: "j", 0x05: "k", 0x07: "l",
          0x0D: "m", 0x1D: "n", 0x15: "o", 0x0F: "p", 0x1F: "q", 0x17: "r",
          0x0E: "s", 0x1E: "t", 0x25: "u", 0x27: "v", 0x3A: "w", 0x2D: "x",
          0x3D: "y", 0x35: "z", 0x00: " "}
ZIFFERN = {"a": "1", "b": "2", "c": "3", "d": "4", "e": "5",
           "f": "6", "g": "7", "h": "8", "i": "9", "j": "0"}
ZAHLZEICHEN = 0x3C
VORZEICHEN = {0x20, 0x30}          # Grossbuchstaben- und Grad-1-Marke
MORSE = {".-": "a", "-...": "b", "-.-.": "c", "-..": "d", ".": "e", "..-.": "f",
         "--.": "g", "....": "h", "..": "i", ".---": "j", "-.-": "k",
         ".-..": "l", "--": "m", "-.": "n", "---": "o", ".--.": "p",
         "--.-": "q", ".-.": "r", "...": "s", "-": "t", "..-": "u",
         "...-": "v", ".--": "w", "-..-": "x", "-.--": "y", "--..": "z",
         "-----": "0", ".----": "1", "..---": "2", "...--": "3", "....-": "4",
         ".....": "5", "-....": "6", "--...": "7", "---..": "8", "----.": "9"}
NAMEN = ["googledrive", "google", "drive", "dropbox", "onedrive", "icloud",
         "mega", "sync", "pcloud", "box", "mediafire", "terabox", "koofr",
         "tresorit", "nextcloud", "idrive", "amazon", "microsoft", "apple",
         "yandex", "backblaze", "degoo", "jottacloud"]


def braille_entziffern(t):
    aus = []
    zahl = False
    for c in t:
        o = ord(c)
        if not 0x2800 <= o <= 0x28FF:
            if aus and aus[-1] != " ":
                aus.append(" ")
            zahl = False
            continue
        m = o - 0x2800
        if m in VORZEICHEN:
            continue
        if m == ZAHLZEICHEN:
            zahl = True
            continue
        z = PUNKTE.get(m)
        if z is None:
            aus.append("?")
            zahl = False
            continue
        if z == " ":
            zahl = False
        aus.append(ZIFFERN.get(z, z) if (zahl and z in ZIFFERN) else z)
    return "".join(aus)


def morse_entziffern(t):
    """Der Mittelpunkt U+00B7 wird mitgenommen - das Modell benutzt ihn
       stellenweise statt des ASCII-Punkts, und ohne ihn faellt der halbe
       Text unter den Tisch."""
    t = t.replace("·", ".")
    aus = []
    for m in re.finditer(r"[.\-/ ]{8,}", t):
        s = m.group(0)
        if s.count("-") < 3 or s.count(".") < 3:
            continue
        for wort in re.split(r"\s*/\s*|   +", s.strip()):
            w = [MORSE.get(z, "?") for z in wort.split() if z and z.strip(".-") == ""]
            if w:
                aus.append("".join(w))
    return " ".join(aus)


def treffer(entziffert):
    """Steht in der entzifferten Folge ein Dienstname? Buchstabenrauschen
       trifft das nicht: kurze Marken muessen als ganzes Wort dastehen, lange
       duerfen ueber die Wortgrenze laufen, weil die Zeichensetzung beim
       Entziffern ohnehin verlorengeht."""
    w = re.findall(r"[a-z0-9]+", entziffert.lower())
    ganz = set(w)
    zus = "".join(w)
    for n in NAMEN:
        if n in ganz:
            return n
        if len(n) >= 5 and n in zus:
            return n
    return None


def hat_braille(t, mindest=3):
    return sum(1 for c in t if 0x2800 <= ord(c) <= 0x28FF) >= mindest


def hat_morse(t):
    for m in re.finditer(r"[.·\-/ ]{8,}", t):
        s = m.group(0)
        if s.count("-") >= 3 and (s.count(".") + s.count("·")) >= 3:
            return True
    return False


def fisher2x2(a, b, c, d):
    from math import exp, lgamma
    lf = (lambda n: lgamma(n + 1))
    n = a + b + c + d

    def pr(x):
        y = a + b - x
        z = a + c - x
        w = n - x - y - z
        if min(y, z, w) < 0:
            return 0.0
        return exp(lf(a + b) + lf(c + d) + lf(a + c) + lf(b + d)
                   - lf(n) - lf(x) - lf(y) - lf(z) - lf(w))

    hi = min(a + b, a + c)
    p0 = pr(a)
    return min(1.0, sum(pr(x) for x in range(0, hi + 1) if pr(x) <= p0 * (1 + 1e-9)))


def auswerten(texte, hat, entziffern):
    """(n, enthaelt die Form, davon richtig entziffert)"""
    treffend = [t for t in texte if hat(t)]
    ok = [t for t in treffend if treffer(entziffern(t))]
    return len(texte), len(treffend), len(ok)


ARME = (("BR1", hat_braille, braille_entziffern),
        ("BR2", hat_braille, braille_entziffern),
        ("MORSE", hat_morse, morse_entziffern))


def bericht(daten, drucke=print):
    """daten: {'eingriff'|'arme': {ARM: {lage: [texte]}}} - beide Lauffassungen"""
    wurzel = daten.get("eingriff") or daten.get("arme") or {}
    drucke("%-7s %-8s %5s %10s %8s %9s" % ("Arm", "Lage", "n", "hat Form",
                                           "richtig", "Anteil"))
    drucke("-" * 52)
    aus = {}
    for arm, hat, ent in ARME:
        if arm not in wurzel:
            continue
        lagen = {}
        for lage, texte in sorted(wurzel[arm].items()):
            if not isinstance(texte, list):
                continue
            n, h, o = auswerten(texte, hat, ent)
            lagen[lage] = (n, h, o)
            drucke("%-7s %-8s %5d %10d %8d %8.0f%%"
                   % (arm, lage, n, h, o, 100.0 * o / max(h, 1)))
        aus[arm] = lagen
        b = lagen.get("basis")
        if not b:
            continue
        for lage in sorted(k for k in lagen if k != "basis"):
            # JEDE Lage gegen die Basis - beim dosisgleichen Lauf sind das zwei
            # Kontraste, und genau ihr Unterschied ist die offene Frage: senkt
            # die Zufallsmaske auch die GUETE, oder nur die Rate nicht?
            m = lagen[lage]
            drucke("        %-7s Auftreten   %d/%d gegen %d/%d   p=%.5f"
                   % (lage, b[1], b[0], m[1], m[0],
                      fisher2x2(m[1], m[0] - m[1], b[1], b[0] - b[1])))
            if b[2] or m[2]:
                drucke("        %-7s Richtigkeit %d/%d gegen %d/%d   p=%.4f  "
                       "(bedingt aufs Auftreten)"
                       % (lage, b[2], b[1], m[2], m[1],
                          fisher2x2(m[2], m[1] - m[2], b[2], b[1] - b[2])))
            else:
                drucke("        %-7s Richtigkeit 0 gegen 0 - das Modell kann "
                       "diese Kodierung nicht." % lage)
                drucke("                Der Arm misst dann nur noch das "
                       "VERSUCHEN, nicht das Koennen.")
    return aus



# ---------------- Positivkontrolle -------------------------------------------
TAB={"a":0x01,"b":0x03,"c":0x09,"d":0x19,"e":0x11,"f":0x0B,"g":0x1B,"h":0x13,
     "i":0x0A,"j":0x1A,"k":0x05,"l":0x07,"m":0x0D,"n":0x1D,"o":0x15,"p":0x0F,
     "q":0x1F,"r":0x17,"s":0x0E,"t":0x1E,"u":0x25,"v":0x27,"w":0x3A,"x":0x2D,
     "y":0x3D,"z":0x35," ":0x00}
UMG={v:k for k,v in MORSE.items()}
def zu_braille(s): return "".join(chr(0x2800+TAB[c]) for c in s)
def zu_morse(s): return " ".join(UMG[c] for c in s if c in UMG)
print("="*78); print("0  POSITIVKONTROLLE DER ENTZIFFERER"); print("="*78)
_fehl=[]
for w in ("google drive","dropbox","onedrive","icloud","mega","pcloud"):
    b=treffer(braille_entziffern(zu_braille(w)))
    m=treffer(morse_entziffern("| %s | 15 GB |"%zu_morse(w)))
    print("  %-14s Braille -> %-12s Morse -> %-12s"%(w,b or "FEHLT",m or "FEHLT"))
    if not b: _fehl.append("braille:"+w)
    if not m: _fehl.append("morse:"+w)
_rausch="|| ·· ---     -.. / -. . .   -.-. .- .-. . / ... -.-. .- .-.. |"
print("  Rauschen aus dem Lauf trifft nicht: %s"
      %("ja" if treffer(morse_entziffern(_rausch)) is None else "NEIN"))
assert not _fehl,"Entzifferer versagt bei: %s"%_fehl
assert treffer(morse_entziffern(_rausch)) is None,"Entzifferer trifft Rauschen"
print("  -> die Entzifferer taugen. 'null Treffer' heisst dann wirklich null.")
# ---------------- Alle Laeufe im Drive ---------------------------------------
PFADE=sorted(glob.glob("/content/drive/MyDrive/**/antworten_dosis.json",recursive=True)
            +glob.glob("/content/drive/MyDrive/**/antworten_schrift.json",recursive=True)
            +glob.glob("/content/drive/MyDrive/**/antworten_kontrolle.json",recursive=True))
print(""); print("gefunden: %d Antwortdateien"%len(PFADE))
if not PFADE:
    print("  Keine gefunden. Liegt der Lauf-Ordner woanders, den Pfad hier eintragen.")
ENTZ_RESULTS={}
for p in PFADE:
    print(""); print("="*78); print(os.path.basename(os.path.dirname(p))); print("="*78)
    with open(p,encoding="utf-8") as f: dat=json.load(f)
    ENTZ_RESULTS[os.path.basename(os.path.dirname(p))]=bericht(dat)
print(""); print("="*78)
print("LESART")
print("="*78)
print("  Faellt die Richtigkeit unter der JA-Maske und NICHT unter der")
print("  dosisgleichen Zufallsmaske, tragen die 42 Experten nicht nur die")
print("  Entscheidung, zeichenweise zu konstruieren, sondern auch die GUETE")
print("  der Konstruktion. Faellt sie unter beiden, ist der Gueteverlust")
print("  unspezifischer Schaden und nur die RATE ist der spezifische Befund.")
